# Stock research multi agent sytem
## Presented by:  
### Tushar Kumar    
### Emmanuel Sadek    
### Juan Norena    
This Final project for the Natural language processing Class AAI-520 consists on a real-world financial analysis system powered by agentic AI. Writes a short research report about a stock in five sections: overview, price behavior, financial health, recent news, conclusion. Numbers come from yfinance, headlines from NewsAPI. A planner, a per-headline prompt chain, up to three analysts and a writer produce the first draft. Then a grader scores it against a checklist and an optimizer revises it, for up to three rounds. Checks the first draft failed are saved to **agent_memory.json** and passed into later runs.

## Agents

Each agent is its own function. ***run_research***, is the coordinator and calls them in the following order:

1. ***planner_agent*** looks at the company's name, sector and industry and picks three of the four data tools plus how much price history to pull. The tools it picks run next, along with the news fetching.
2. ***article_agent*** runs each headline through a three-step chain which include ***classify_article***, ***extract_facts***, ***summarize_facts***.
3. ***route_articles*** hands each classified headline to the analyst for its category.
4. ***analyst_agent*** runs for every analyst in ***SPECIALISTS*** which include market, financial, and news focuses and returns 2-3 findings.
5. ***writer_agent*** writes the report from the findings and the raw data.
6. ***evaluator_agent*** grades the report and ***optimizer_agent*** rewrites whatever failed, for up to three rounds during the self-review step. The system grades its own draft and fixes it.


In [ ]:
import os, re, json
from datetime import datetime, timedelta
import requests
import yfinance as yf
from openai import OpenAI
from google.colab import userdata

#Set keys for APIs
client = OpenAI(api_key=userdata.get("OPENAI_API_KEY"))
NEWS_API_KEY = userdata.get("NEWS_API_KEY")


# define gpt-4o-mini as model
def call_llm(prompt, temp=0.2, model="gpt-4o-mini"):
    resp = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=temp,
        response_format={"type": "json_object"},
    )
    return json.loads(resp.choices[0].message.content)

## Data

***get_news*** accesses NewsAPI endpoint for the companys ticker, combined with at least one of earnings, stock, revenue, shares or market. It requests articles from the last 21 days, sorts results by relevancy, and takes 10.

***preprocess*** collapses whitespace, drops exact duplicates and skips anything containing "Removed".

In [ ]:
# headlines from the last 21 days containing earnings, stock, revenue, shares or market info
def get_news(stock):
    name = stock.info["longName"]
    params = {
        "q": f'("{name}" OR "{stock.ticker}") AND (earnings OR stock OR revenue OR shares OR market)',
        "from": (datetime.now() - timedelta(days=21)).strftime("%Y-%m-%d"),
        "sortBy": "relevancy",
        "pageSize": 10,
        "language": "en",
        "apiKey": NEWS_API_KEY,
    }
    res = requests.get("https://newsapi.org/v2/everything", params=params).json()
    if res["status"] != "ok":
        print("NewsAPI error:", res["message"])

    articles = []
    for a in res.get("articles", []):
        articles.append(f"{a['title']}. {a['description'] or ''}")
    return articles


# clean whitespace, drop duplicates and skip deleted ones
def preprocess(articles):
    clean = []
    for text in articles:
        text = " ".join(text.split())
        if text not in clean and "[Removed]" not in text:
            clean.append(text)
    return clean

## Tools

Four data tools to handle yfinance data. ***TOOLS*** holds a one-line description of each, and that description is all the planner knows about a tool. ***run_tool*** directs the chosen name back to its function.

- ***get_prices*** pulls daily closes for the planner's period and returns the latest close, the total return from first to last close, annualized volatility and max drawdown.
- ***get_financials*** gets revenue, growth, net income, net margin and free cash flow from annual statements, and debt-to-equity and current ratio from the balance sheet.
- ***get_valuation*** returns trailing and forward P/E.
- **get_earnings** takes the most recent quarter and returns the EPS estimate, the actual, and the surprise.

***run_tool*** catches any exception, so the run goes on without that tool.

Each value a tool returns is analyzed for "what does this mean for an investor".

In [ ]:
# Tool description to be fed into planner
TOOLS = {
    "prices": "price return, volatility and max drawdown over the period you pick",
    "financials": "revenue growth, margins, free cash flow, debt and liquidity",
    "valuation": "trailing and forward P/E",
    "earnings": "last quarter's EPS compared to the analyst estimate",
}


# Tool to return stock price history
def get_prices(stock, period):
    close = stock.history(period=period)["Close"].dropna()
    returns = close.pct_change().dropna()
    return {
        "period": period,
        "latest_close_usd": round(close.iloc[-1], 2),
        "price_return_pct": round((close.iloc[-1] / close.iloc[0] - 1) * 100, 1),
        "annualized_volatility_pct": round(returns.std() * 252 ** 0.5 * 100, 1),
        "max_drawdown_pct": round((close / close.cummax() - 1).min() * 100, 1),
    }

# Tool to return financial information
def get_financials(stock):
    inc = stock.income_stmt
    bal = stock.quarterly_balance_sheet
    inc_now = inc.iloc[:, 0]
    bal_now = bal.iloc[:, 0]
    cf_now = stock.cashflow.iloc[:, 0]
    rev = inc.loc["Total Revenue"]
    rev_now, rev_prev = rev.iloc[0], rev.iloc[1]
    ni = inc_now["Net Income"]
    return {
        "fiscal_year_end": str(inc.columns[0].date()),
        "revenue_billion_usd": round(rev_now / 1000000000, 1),
        "revenue_growth_pct": round((rev_now / rev_prev - 1) * 100, 1),
        "net_income_billion_usd": round(ni / 1000000000, 1),
        "net_margin_pct": round(ni / rev_now * 100, 1),
        "balance_sheet_date": str(bal.columns[0].date()),
        "debt_to_equity": round(bal_now["Total Debt"] / bal_now["Stockholders Equity"], 2),
        "current_ratio": round(bal_now["Current Assets"] / bal_now["Current Liabilities"], 2),
        "free_cash_flow_billion_usd": round(cf_now["Free Cash Flow"] / 1000000000, 1),
    }

# Tool to return P/E ratios
def get_valuation(stock):
    info = stock.info
    return {"trailing_pe": round(info["trailingPE"], 1), "forward_pe": round(info["forwardPE"], 1)}


# Tool to return actual vs expected EPS
def get_earnings(stock):
    hist = stock.earnings_history.sort_index()
    last = hist.iloc[-1]
    return {
        "quarter_end": str(hist.index[-1].date()),
        "eps_estimate": round(last["epsEstimate"], 2),
        "eps_actual": round(last["epsActual"], 2),
        "eps_surprise_pct": round((last["epsActual"] / last["epsEstimate"] - 1) * 100, 1),
    }


# runs a tool by name, if it errors out it returns {} so the run continues
def run_tool(name, stock, plan):
    try:
        if name == "prices":
            return get_prices(stock, plan["period"])
        elif name == "financials":
            return get_financials(stock)
        elif name == "valuation":
            return get_valuation(stock)
        elif name == "earnings":
            return get_earnings(stock)
    except Exception as e:
        print(f"{name} tool failed, skipping it: {e}")
    return {}

## Planner

***gather*** calls the planner for each ticker. ***planner_agent*** gets the profile which is made up of the name sector and industry of the ticker. The ***TOOLS*** descriptions and any lessons learned from previous runs. It then  has to return the three tools that matter most for this company, a price window and a reason why it choose what it did. ***get_news*** runs every time because the article chain needs it.

In [ ]:
# planner: decides which tools to run after examing company profile
def planner_agent(ticker, profile, lessons):
    prompt = (
        f"We're putting together a short research report on {ticker}, covering how the stock has been moving, "
        f"how healthy the company's finances are, and what's been in the news lately. Some info on the company: {profile}\n"
        f"Data tools we can run: {TOOLS}\n"
        "Decide which three tools matter most for this company right now, and how much price history "
        "we should use ('6mo', '1y' or '2y').\n"
        f"Things our earlier reports missed: {lessons or 'nothing'}\n"
        "Please answer in JSON with 'steps' (a list of tool names), 'period', and 'reason'."
    )
    plan = call_llm(prompt)

    # keep real tools. anything other than 6mo/1y/2y defaults to 1y
    plan["steps"] = [t for t in TOOLS if t in str(plan["steps"]).lower()]
    if plan["period"] not in ["6mo", "1y", "2y"]:
        plan["period"] = "1y"
    return plan

## Article agent (prompt chain)

Up to three calls per headline, each its own function

1. ***classify_article*** labels the first 600 characters as earnings, market, news or irrelevant.
2. ***extract_facts*** lists the key facts from the same 600 characters. **EXTRACT_FOCUS** says what to look for in each category
3. ***summarize_facts*** turns the facts into one sentence.

The summary step only gets the facts list. The label also picks the analyst.

In [ ]:
# define extractable information for each catagory
EXTRACT_FOCUS = {
    "earnings": "reported results, guidance, and beats or misses",
    "market": "price moves, analyst ratings or price targets, and trading activity",
    "news": "the event, product, or deal, and who is involved",
}

# Call 1: classify article based on content
def classify_article(text, ticker):
    instruction = (
        f"Which category fits this article about {ticker}?\n"
        "- earnings: results, guidance, EPS, revenue\n"
        "- market: stock price moves, analyst price targets, trading volume\n"
        "- news: other company events like products, deals, lawsuits, or people\n"
        f"- irrelevant: not really about {ticker}\n\n"
        f"Article: {text[:600]}\n\n"
        "Reply in JSON with a 'category' key."
    )
    return call_llm(instruction, 0)["category"].lower()

# Call 2: Extract key information
def extract_facts(text, ticker, cat):
    instruction = (
        f"List the key facts in this {cat} article about {ticker}, mainly {EXTRACT_FOCUS[cat]}. "
        "Keep numbers exactly as written and don't add anything that isn't in the article.\n\n"
        f"Article: {text[:600]}\n\n"
        "Reply in JSON with a 'facts' key (a list of short strings)."
    )
    return call_llm(instruction, 0)["facts"]

# Call 3: Condense key infomration into 1 sentence
def summarize_facts(facts):
    instruction = (f"Summarize these facts in one sentence for an investor: {facts}\n"
                   "Reply in JSON with a 'summary' key.")
    return call_llm(instruction, 0)["summary"]

# Define Article Agent
def article_agent(text, ticker):
    cat = classify_article(text, ticker)
    article = {"text": text, "category": cat}
    if cat in EXTRACT_FOCUS:
        article["facts"] = extract_facts(text, ticker, cat)
        article["summary"] = summarize_facts(article["facts"])
    return article

## Specialist analysts + router

***SPECIALISTS*** defines three analysts a focus string, which tools output the analyst gets, and which article labels it handles.

***route_articles*** is the router. Each article is added to every analyst whose handles include its label.

***analyst_agent*** prompts one analyst with its tool data, the summaries of the first five articles routed to it and the lessons, and asks for 2-3 findings.

In [ ]:
# three specialists with focus, tool data, and the news categories the router sends it
SPECIALISTS = {
    "market": {
        "focus": "price trend, volatility and drawdown, valuation, and which "
                 "news explains the price moves",
        "tools": ["prices", "valuation"],
        "handles": ["market"],
    },
    "financial": {
        "focus": "revenue growth, margins, debt, liquidity and free cash "
                 "flow, plus the last earnings surprise and any earnings news",
        "tools": ["financials", "earnings"],
        "handles": ["earnings"],
    },
    "news": {
        "focus": "what each event means for revenue, sentiment or risk",
        "tools": [],
        "handles": ["news"],
    },
}


# router sends each article to its category's analyst, irrelevant ones get dropped
def route_articles(articles):
    routed = {name: [] for name in SPECIALISTS}
    for a in articles:
        for name, spec in SPECIALISTS.items():
            if a["category"] in spec["handles"]:
                routed[name].append(a)
    return routed


# # runs one specialist 2-3 findings from tool data and up to 5 article summaries
def analyst_agent(name, metrics, articles, lessons):
    news = [a["summary"] for a in articles[:5]]
    prompt = (
        f"You're the {name} analyst on a stock research team. "
        f"Focus on {SPECIALISTS[name]['focus']}. "
        "Using only the data below, write 2-3 short findings about what it means for an investor.\n\n"
        f"Metrics: {json.dumps(metrics)}\nNews: {news}\n\n"
        f"Things earlier reports missed: {lessons or 'nothing yet'}\n"
        "Reply in JSON with a 'findings' key (a list of strings)."
    )
    return call_llm(prompt)["findings"]

## Writer

**writer_agent** recieves the analysts findings plus the full ***data***, profile, and article summaries. The prompt asks for five sections to be written. Lessons are added as things earlier reports missed.

The saved first drafts ran 243 to 390 words, all under the grader's 400-word cap.

In [ ]:
# writer agent - turns all the findings into the report
def writer_agent(ticker, findings, data, lessons):
    prompt = (
        f"Please write a short research report on {ticker}, split into these sections: "
        "overview, price behavior, financial health, recent news, and add a conclusion. "
        "Work using the analysts findings below, and only use numbers that actually appear in the data. "
        "If a section has no data, say so in one sentence instead of making something up or guessing.\n"
        f"Please be sure to include the listed things that our earlier reports missed: {lessons or 'nothing so far'}\n\n"
        f"Findings: {json.dumps(findings)}\n\n"
        f"Data: {json.dumps(data)}\n\n"
        "Please answer in JSON with a 'report' key that holds the whole report as one string, "
        "with a blank line between sections."
    )
    return call_llm(prompt, 0.2)["report"]

## Memory

***MEMORY_FILE*** Contains the exact text of a check of the first draft which failed. ***save_lessons*** appends each failed check that isn't already in the file. Nothing is ever removed, so the list only grows. Only the first draft counts.

In [ ]:
# Memory file contains lists of failed checks, reset every time notebook is run
MEMORY_FILE = "agent_memory.json"

def load_lessons():
    if os.path.exists(MEMORY_FILE):
        with open(MEMORY_FILE) as f:
            return json.load(f)
    return []

# add every check that failed
def save_lessons(checks, passed):
    lessons = load_lessons()
    for check, ok in zip(checks, passed):
        if not ok and check not in lessons:
            lessons.append(check)
    with open(MEMORY_FILE, "w") as f:
        json.dump(lessons, f, indent=2)

## Evaluator-optimizer loop

***run_research*** builds the checklist which includes one check per metric key not in ***skip***, and 15 ***general_checks***.

certain checks are simple pythonic checks that evaluate word count, regex, and precesce of certain words

The grader sees the report but never ***data***, and must quote the sentences that satisfy it. A pass needs a yes and at least one quote. if the check fails, those quotes go to the optimizer

***optimizer_agent*** is prompted to edit only what the failed checks need. Its printed change list is its own account.

The loop keeps a revision only if it scores higher. A rejected round leaves the same report and fixes for the next one.

In [ ]:
# the checklist for gpt-4o to grade
general_checks = [
    "Clearly says if the positives outweigh the risks or the other way around",
    "Uses two different metrics together in one sentence (e.g. free cash flow and current ratio)",
    "Admits what the analysis is missing (e.g. no competitor data)",
    "Explains how the industry affects the outlook instead of just naming it",
    "The conclusion gives the two main reasons for the buy/hold/sell call, and nothing else in the report contradicts it",
    "Gives at least two risks, each backed up by a number from the data (e.g. max drawdown, debt-to-equity)",
    "Names something to watch with a concrete trigger, like a number or the next earnings date",
    "Ties at least one headline to an actual number from the data",
    "Points out where the signals disagree (e.g. strong revenue growth but the stock is falling)",
    "Only uses numbers that are actually in the data",
    "Says which fiscal year the financials are from",
    "Stays under 400 words",
]

# regex for certain words + explainer
word_checks = {
    "The conclusion uses the word buy, hold or sell for the call": r"\b(buy|hold|sell)\b",
    "Gives a confidence level (low, medium or high)": r"(low|medium|high)[ -]confidence|confidence[^.]*\b(low|medium|high)\b",
    "Gives a time frame (short term or long term)": r"(short|long)[ -]term",
}
general_checks += list(word_checks)

# skip per-metric check
skip = ["period", "latest_close_usd", "fiscal_year_end", "revenue_billion_usd", "net_income_billion_usd",
        "balance_sheet_date", "quarter_end", "eps_estimate", "eps_actual"]

In [ ]:
# grades a report against the checklist, pass/fail for each check plus the overall score
def evaluator_agent(report, data, checks):
    # number check: every $, % and decimal in the report has to show up somewhere in the data
    dataNums = [float(n) for n in re.findall(r"\d+\.?\d*", json.dumps(data).replace(",", ""))]
    badNums = []
    for n in re.findall(r"\$\d[\d,]*\.?\d*|\d+\.?\d*%|\d+\.\d+", report):
        if float(n.strip("$%.").replace(",", "")) not in dataNums:
            badNums.append(n)

    # letters and numbers only to ignroe punctuation
    flat = re.sub(r"[^a-z0-9]", "", report.lower())

    passed, fixes = [], []
    for check in checks:
        fix = check
        if "Only uses numbers" in check:
            ok = not badNums
            fix = f"Remove or correct these numbers, they aren't in the data: {badNums}"
        elif "fiscal year" in check:
            # latest fiscal year has to appear in the report
            fin = data["metrics"].get("financials")
            ok = not fin or fin["fiscal_year_end"][:4] in report
        elif "under 400 words" in check:
            ok = len(report.split()) < 400
            fix = f"Cut the report to under 400 words (it has {len(report.split())}) without dropping anything the other checks need"
        elif check in word_checks:
            # buy/hold/sell check only applied to conclusion
            text = report.lower()
            if "conclusion" in check:
                text = text.split("conclusion")[-1]
            ok = re.search(word_checks[check], text) is not None
        else:
            prompt = (
                f"You are grading one specific thing in a stock report: {check}\n\n"
                f"Here's the report:\n{report}\n\n"
                "Just judge whether the report actually does this, not whether you agree with what it says. "
                "Before you decide, copy out the sentences that do it, word for word (use an empty list if there aren't any).\n"
                "Please answer in JSON with 'quotes' (a list) and 'passed' (true or false)."
            )
            result = call_llm(prompt, 0, "gpt-4o")

            # pass needs real quotes; optimizer sees them
            quotes = [q for q in result["quotes"] if re.sub(r"[^a-z0-9]", "", q.lower()) in flat]
            ok = result["passed"] and len(quotes) > 0 and len(quotes) == len(result["quotes"])
            if quotes:
                fix = f"{check} (sentences the grader looked at: {quotes})"

        passed.append(ok)
        if not ok:
            fixes.append(fix)

    return {"passed": passed, "fixes": fixes, "score": round(100 * sum(passed) / len(passed))}


# optimizer rewrites the report to fix failures
def optimizer_agent(report, fixes, data):
    prompt = (
        "Edit this stock report so it meets every requirement below. "
        "Only change or add the sentences you need to and leave the rest alone. "
        "Only use numbers and facts from the data.\n\n"
        f"Requirements: {fixes}\n\n"
        f"Data: {json.dumps(data)}\n\n"
        f"Report:\n{report}\n\n"
        "Reply in JSON with 'report' (the full edited report as one string) "
        "and 'changes' (a list of what you changed)."
    )
    result = call_llm(prompt)

    # changes only used for this printout
    print("Changes made:")
    for c in result["changes"]:
        print("  -", c)
    return result["report"]

## Pipeline

***gather*** covers everything before the analysts. Stock profile, planner, chosen tools, news cleanup, the article chaining, and routing. ***Gather*** prints the plan, the article counts, It returns ***data*** which include the profile, metrics, and summaries of the routed articles.

***run_research*** loads lessons, calls **gather** runs the analysts, calls the writer, generates a checklist, and performs grades. It also runs the revision loop and saves the first draft's failures.

**use_memory=False** ignores the lessons file and **save_memory=False** leaves it untouched.

In [ ]:
# planner + tools + article agent + router
def gather(ticker, lessons, show_chain=True):
    stock = yf.Ticker(ticker)
    info = stock.info
    profile = {"name": info["longName"], "sector": info["sector"], "industry": info["industry"]}

    plan = planner_agent(ticker, profile, lessons)
    print(f"Plan: {plan['steps']}, {plan['period']} of prices. {plan['reason']}")

    # run only the tools the planner picked, news always runs
    metrics = {}
    for name in plan["steps"]:
        metrics[name] = run_tool(name, stock, plan)
    texts = preprocess(get_news(stock))

    # prompt chain on every headline, route each one to its analyst
    articles = [article_agent(text, ticker) for text in texts]
    routed = route_articles(articles)
    relevant = routed["market"] + routed["financial"] + routed["news"]

    print(f"Articles: {len(texts)} | market: {len(routed['market'])}, financial: {len(routed['financial'])}, news: {len(routed['news'])}")
    if relevant and show_chain:
        print("Chain example:")
        for key in ["text", "category", "facts", "summary"]:
            print(f"  {key}: {relevant[0][key]}")

    data = {"company": profile, "metrics": metrics, "articles": [a["summary"] for a in relevant]}
    return data, routed


# coordinator runs agents in order on gather's output
def run_research(ticker, max_rounds=3, use_memory=True, save_memory=True, inputs=None):
    lessons = load_lessons() if use_memory else []
    print(f"Researching {ticker}... (using {len(lessons)} lessons from past runs)")
    if inputs is None:
        inputs = gather(ticker, lessons)
    data, routed = inputs
    metrics = data["metrics"]

    # analyst only runs if it has tool data or articles
    findings = {}
    for name, spec in SPECIALISTS.items():
        toolData = {}
        for t in spec["tools"]:
            if metrics.get(t):
                toolData[t] = metrics[t]
        if toolData or routed[name]:
            findings[name] = analyst_agent(name, toolData, routed[name], lessons)

    report = writer_agent(ticker, findings, data, lessons)

    # one check per metric + general
    checks = []
    for group in metrics.values():
        for k in group:
            if k not in skip:
                checks.append(f"Says what {k} means for an investor (is it a strength or a concern?)")
    checks += general_checks

    review = evaluator_agent(report, data, checks)
    firstReport, firstReview = report, review
    scores = [review["score"]]
    print(f"\nFirst draft: {sum(review['passed'])}/{len(checks)} checks passed (score {review['score']})")

    # evaluator-optimizer loop revision is only kept if it scores higher
    for i in range(1, max_rounds + 1):
        if review["score"] == 100:
            break
        print(f"\nRound {i} fixes:")
        for fix in review["fixes"]:
            print("  -", fix)
        newReport = optimizer_agent(report, review["fixes"], data)
        newReview = evaluator_agent(newReport, data, checks)
        scores.append(newReview["score"])
        print(f"Round {i} score: {review['score']} -> {newReview['score']}")
        if newReview["score"] > review["score"]:
            report, review = newReport, newReview

    # lessons come from the first draft only
    if save_memory:
        save_lessons(checks, firstReview["passed"])

    return {"checks": checks, "scores": scores, "first_report": firstReport, "first_review": firstReview,
            "report": report, "review": review, "lessons": len(lessons)}

## AAPL example

Deletes ***agent_memory.json*** so we can start fresh without any lessons in this notebook, then runs ***run_research("AAPL")***. The cell prints both reports and provides one pass/fail line per check and the score from every evaluation.

In the saved run the first draft passed 12 of 24 checks, a score of 50. Round 1 brought it to 88 and round 2 to 96. Round 3 scored 92 and was thrown out, so the report printed as optimized is the report generated in round 2.

That report fails one check, which happens to be the 400 word cap. The first draft was 316 words; sequential rounds increased it past 400. 10 checks were fixed by the end of the round.

The first draft's 12 failures are the lessons the memory test starts with.

In [ ]:
# start with an empty memory file
if os.path.exists(MEMORY_FILE):
    os.remove(MEMORY_FILE)

result = run_research("AAPL")

print(f"\n--- ORIGINAL REPORT (score {result['first_review']['score']}/100) ---\n")
print(result["first_report"])
print(f"\n--- OPTIMIZED REPORT (score {result['review']['score']}/100) ---\n")
print(result["report"])
print("\nChecklist (first draft -> final):")

for check, before, after in zip(result["checks"], result["first_review"]["passed"], result["review"]["passed"]):
    print(f"  {'PASS' if before else 'FAIL'} -> {'PASS' if after else 'FAIL'}  {check[:100]}")

print("\nScores:", result["scores"])

Researching AAPL... (using 0 lessons from past runs)
Plan: ['prices', 'financials', 'earnings'], 1y of prices. AAPL is a major player in the technology sector, and analyzing the stock's price movements over the past year will provide insights into its recent performance and volatility. Financial health indicators such as revenue growth, margins, and free cash flow are crucial for understanding the company's financial stability, especially in a competitive industry. Lastly, examining the latest earnings report and comparing EPS to analyst estimates will help gauge market expectations and the company's ability to meet them.
Articles: 9 | market: 2, financial: 0, news: 4
Chain example:
  text: Apple gets a $360 price target from JP Morgan despite iPhone demand concerns. Just days after Apple’s stock took a hit over reports of iPhone 18 Pro production cuts, JP Morgan has raised its AAPL price target by $20. Here are the details. more…
  category: market
  facts: ['Apple gets a $360 price t

## Memory test

Tests whether lessons from one company improve the first draft for another and that the memory functionality is working properly. MSFT, JNJ and NVDA each get two full runs (first draft plus up to three rounds), which include one without memory and one with memory.

**gather** runs once per ticker with an empty lessons list and **show_chain=False**, and both of that ticker's runs reuse the result through **inputs**, so they share the plan, tool data and article summaries. The three no-memory runs (**use_memory=False**, **save_memory=False**) go first and leave the file untouched, then the three memory runs each read the file and add their own first-draft failures.

The only input that changes between a ticker's two runs is the lessons in the analyst and writer prompts. Outputs can still drift due to LLM randomness.

each run's scores along with the average first-draft score per condition and the final lessons list as displayed below.

In [ ]:
# each ticker runs once without memory, then all three run again with memory
tickers = ["MSFT", "JNJ", "NVDA"]
show_drafts = True  # False -> don't print the reports

# plan, data and article summaries get made once per ticker (no lessons) and both runs reuse them
inputs = {}
for t in tickers:
    print(f"\n========== {t}: plan and data ==========")
    inputs[t] = gather(t, [], show_chain=False)

# without memory = doesn't read or write the lessons file
no_mem, with_mem = {}, {}
for t in tickers:
    print(f"\n========== {t} WITHOUT MEMORY ==========")
    no_mem[t] = run_research(t, use_memory=False, save_memory=False, inputs=inputs[t])
    if show_drafts:
        print(f"\n--- ORIGINAL REPORT (score {no_mem[t]['scores'][0]}/100) ---\n\n{no_mem[t]['first_report']}")
        print(f"\n--- FINAL REPORT (score {no_mem[t]['review']['score']}/100) ---\n\n{no_mem[t]['report']}")

# with memory = reads the lessons so far (starting with AAPL's) and adds its own,
# so JNJ also gets MSFT's lessons and NVDA gets both
for t in tickers:
    print(f"\n========== {t} WITH MEMORY ==========")
    with_mem[t] = run_research(t, inputs=inputs[t])
    if show_drafts:
        print(f"\n--- ORIGINAL REPORT (score {with_mem[t]['scores'][0]}/100) ---\n\n{with_mem[t]['first_report']}")
        print(f"\n--- FINAL REPORT (score {with_mem[t]['review']['score']}/100) ---\n\n{with_mem[t]['report']}")


========== MSFT: plan and data ==========
Plan: ['prices', 'financials', 'earnings'], 1y of prices. Using a 1-year period provides a balanced view of Microsoft's stock performance, capturing recent trends while also accounting for any significant market events. The selected tools focus on price movements, financial health, and earnings performance, which are crucial for understanding the company's current standing and future outlook in the technology sector.
Articles: 9 | market: 1, financial: 0, news: 0

========== JNJ: plan and data ==========
Plan: ['prices', 'financials', 'earnings'], 1y of prices. A 1-year period provides a balanced view of JNJ's stock performance, capturing recent trends while also reflecting any longer-term impacts from market conditions and company-specific events. The selected tools focus on price movements, financial health, and earnings performance, which are crucial for assessing the company's current standing and future outlook in the healthcare sector.
A

In [ ]:
# scores after each round, first draft first. if a number doesn't go up, that revision got
# thrown out and the next round tried again from the better version
first_off, first_on = 0, 0
for t in tickers:
    print(t)
    print("  without memory:", " > ".join(str(s) for s in no_mem[t]["scores"]))
    print("  with memory:   ", " > ".join(str(s) for s in with_mem[t]["scores"]), f"(used {with_mem[t]['lessons']} lessons)")
    first_off += no_mem[t]["scores"][0]
    first_on += with_mem[t]["scores"][0]

print(f"\nAverage first draft: {first_off / len(tickers):.0f} without memory, {first_on / len(tickers):.0f} with memory")
print("\nLessons in memory now (AAPL's plus what the memory runs added):")
for lesson in load_lessons():
    print("  -", lesson)

MSFT
  without memory: 58 > 88 > 88 > 88
  with memory:    71 > 83 > 92 > 96 (used 12 lessons)
JNJ
  without memory: 62 > 83 > 92 > 92
  with memory:    92 > 92 > 92 > 96 (used 17 lessons)
NVDA
  without memory: 67 > 92 > 92 > 92
  with memory:    88 > 100 (used 17 lessons)

Average first draft: 62 without memory, 84 with memory

Lessons in memory now (AAPL's plus what the memory runs added):
  - Says what max_drawdown_pct means for an investor (is it a strength or a concern?)
  - Says what free_cash_flow_billion_usd means for an investor (is it a strength or a concern?)
  - Says what eps_surprise_pct means for an investor (is it a strength or a concern?)
  - Clearly says if the positives outweigh the risks or the other way around
  - Admits what the analysis is missing (e.g. no competitor data)
  - Explains how the industry affects the outlook instead of just naming it
  - The conclusion gives the two main reasons for the buy/hold/sell call, and nothing else in the report contradicts 

## Findings

All seven runs finished between 88 and 100, and the revision loop raised the score in every one. AAPL went from 50 to 96.

For the stocks we test, With memory enabled, first drafts passed 17, 22 and 21 of 24 checks, against 14, 15 and 16 with memory disabled. Final scores increased from 88, 92 and 92 to 96, 96 and 100. These scores demonstrate improvement accross the board.

Of the 19 checks that flipped to pass, 12 are the same four, fiscal year, buy/hold/sell, and confidence level. All four were AAPL lessons and failed in every no-memory first draft. Memory didn't fix the industry check, which failed in all three memory first drafts and shows up in 13 of the 19 fix lists.

Caveats: JNJ and NVDA had 17 lessons to MSFT's 12, and with one run each there's no separating. Since the lessons are the checklist, this mostly shows that handing the checklist to the analysts and writer raises the checklist score. The file ended at 17 lessons learned, you can attribute 12 to APPL and 5 from MSFT.


Observations and room for improvement.
- The loop makes length worse. First drafts were 243 to 390 words, but four of seven finals are over 400, even after rounds where the optimizer said it cut the report.
- Examples in the check text get copied: every passing report names competitor data as the missing piece.
- JNJ's no-memory final scored 92 while calling a 35.2% return "significantly outperforming the S&P 500, which returned 52%" which reminds us LLM's are not perfect.
- The grader isn't completely consistent. JNJ's no-memory draft passed the **max_drawdown_pct** check, then failed it after round 1 on an identical sentence.
- The planner picked the same tools for every ticker, and no headline was ever routed to the financial analyst. This could a random occurance, artifact of their being less financial articles, or potentially the agent is more likely to decide that financial data is least improtant.  

### *AI Disclosure*

*AI tools were used to assist with debugging, syntax clarification, and outlining initial code structures that were then reviewed, modified, and integrated by ourselves. All final implementation, analysis, and decisions are our own work and reflect our understanding of the material.*